# 🧪 GIADA — prerequisiti IV-B1 e IV-B2 per la Task 32
Prima CaDynamics_E2 con corrente di calcio imposta; solo se passa, CaDynamics → SK_E2 con voltaggio imposto. Nessun training né feedback elettrico chiuso in questo notebook.


In [ ]:
from pathlib import Path
import base64,json,os,shutil,subprocess,sys,uuid
from IPython.display import display,Javascript
WORK=Path('/kaggle/working')/('giada_iv_b_'+uuid.uuid4().hex[:8]);WORK.mkdir()
REPO=WORK/'giada';TEACHER=WORK/'neuron_as_deep_net'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin',os.environ.get('GIADA_CODE_REVISION','codex/surrogate-validity-audit')],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
try:
 import neuron
except ImportError:
 subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
 import neuron
assert neuron.__version__.split('+')[0]=='8.2.7',neuron.__version__
print({'revision':REVISION,'teacher':'074c466','neuron':neuron.__version__})


In [ ]:
OUTPUT=Path('/kaggle/working/artifacts')/('giada_iv_b_'+REVISION[:7]+'_'+uuid.uuid4().hex[:8])
completed=subprocess.run([sys.executable,'-u',str(REPO/'scripts/run_iv_b_calcium_prerequisite.py'),'--output',str(OUTPUT),'--teacher',str(TEACHER)],check=False)
path=OUTPUT/('failure_report.json' if (OUTPUT/'failure_report.json').exists() else 'final_report.json')
report=json.loads(path.read_text()) if path.is_file() else {'valid':False,'error':f'worker interrupted: {completed.returncode}'}
display({k:report.get(k) for k in ('valid','iv_b1_valid','iv_b2_valid','task32_feedback_authorized','b1_episode_count','b2_episode_count','perturbation_informative','error')})
if report.get('b1_rows'): display({'b1_worst_rmse_mM':max(r['rmse'] for r in report['b1_rows']),'b1_worst_abs_mM':max(r['max_abs'] for r in report['b1_rows'])})
if report.get('b2_rows'): display({'b2_worst_cai_rmse_mM':max(r['cai']['rmse'] for r in report['b2_rows']),'b2_worst_sk_rmse':max(r['sk_composed']['rmse'] for r in report['b2_rows'])})
print({'output':str(OUTPUT),'exit':completed.returncode})


In [ ]:
archive=Path(shutil.make_archive(str(Path('/kaggle/working')/OUTPUT.name),'zip',OUTPUT))
if os.environ.get('GIADA_MANUAL_DOWNLOAD','0')=='1':
 payload=base64.b64encode(archive.read_bytes()).decode('ascii')
 display(Javascript(f"const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"))
print({'archive':str(archive),'size_mib':round(archive.stat().st_size/2**20,2)})
